# 根据种子数据扩增&数据合成

In [1]:
from importlib.metadata import version

print(f"openai version:  {version("openai")}")
print(f"python-dotenv version:  {version("python-dotenv")}")

openai version:  2.30.0
python-dotenv version:  1.1.0


In [2]:
import os
import re
import pickle
import random
import time
import threading
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from dotenv import load_dotenv
from openai import OpenAI

# 加载环境变量
load_dotenv()

random.seed(42)

# 提示词

In [3]:
QUERY_EXPANSION_PROMPT = """
你是一个经验丰富的文字改写专家，对中国的旅游城市，交通，景点路线等非常熟悉。你的任务是根据输入的用户关于旅游的问题，扩充生成5个新的问题。

以下是一些生成新问题的要求：
1.给的用户问题只是参考，要输出完全新的问题，不能跟原有的句子意思相同或相近。
2.新的问题里面出现的地名，景点，必须是真实的，问题是符合逻辑的，请结合你的只旅游知识来进行改写，可以适当替换里面出现的地点/城市。
3.(可选)如果有必要，可以在提问里面增加一些约束，例如价格，距离，评分，交通方式，预算，途经点等。

请直接输出改写后的句子，每一个句子是一行，不要输出理由或者其他无关内容。

用户问题：{}
"""

# 定义 OpenAI Client

In [4]:
llm_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"]
)

# 写调用逻辑

In [24]:
def chat(prompt, max_retry=3):
    def do_chat(text):
        prompt = QUERY_EXPANSION_PROMPT.format(text)
        completions = llm_client.chat.completions.create(
            model="deepseek-v4-pro",
            messages=[
                {"role": "system", "content": "你是有用的人工智能助手。"},
                {"role": "user", "content": prompt}
            ]
        )
        return text, completions.choices[0].message.content

    while max_retry > 0:
        try:
            return do_chat(prompt)
        except Exception as e:
            max_retry -= 1
            sleep_seconds = random.randint(1, 4)
            time.sleep(sleep_seconds)
    return  

# 测试一下

In [25]:
result = chat("五一假期从重庆到宜昌3日高铁，途经恩施，自然风光和土家族文化，带小孩，预算总6000，推荐吊脚楼。")

In [26]:
print(result[1])

国庆期间从成都到张家界4日自驾游，途经凤凰古城，体验苗族风情和喀斯特地貌，带上父母，总预算8000元，推荐特色民宿。
端午假期从武汉到庐山3日动车出行，路过九江，观赏瀑布和云海景观，情侣出游，人均花费控制在1500元，求性价比高的山景客栈。
春节前后从广州到桂林4日高铁游，顺道游览阳朔，带学龄儿童欣赏喀斯特山水，家庭总预算10000元，寻找适合亲子的田园民宿。
暑期从杭州到黄山3日自由行，乘坐高铁经绩溪，想领略徽派建筑和云海日出，一家三口预算5000元，请推荐山顶酒店或特色民居。
中秋假期从西安到兰州4日火车旅行，途经天水麦积山，探索石窟艺术和黄河文化，带老人出行，全程花费不超7000元，寻求安静的四合院住宿。


# 并发调大模型

In [28]:
MAX_WORKERS = 50        # 并发度 50，可根据实际的大模型接口 QPS 调整

input_path = "../data/train_seed.jsonl"
output_path = "../data/train_expansion.jsonl"


queries = []
with open(input_path) as fd:
    for idx, line in enumerate(fd):
        info = json.loads(line)
        queries.append((idx, info["question"]))

expand_queries = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {idx: executor.submit(chat, query) for idx, query in queries}
    for idx in tqdm(futures):
        future = futures[idx]
        result = future.result()
        if result is None:
            continue
        query, expand = result
        expand = expand.split("\n")
        expand = [re.sub(r"\d. ", "", item) for item in expand if item.strip()]
        expand_queries.append(query)
        expand_queries.extend(expand)

# 去重
expand_queries = list(set(expand_queries))

with open(output_path, "w") as fw:
    for query in expand_queries:
        info = {"question": query}
        fw.write(json.dumps(info, ensure_ascii=False) + "\n")

  0%|          | 0/10 [00:00<?, ?it/s]

In [30]:
! cat ../data/train_expansion.jsonl | shuf -n 10

{"question": "我想了解从张家界国家森林公园门票站进，游览金鞭溪、袁家界、杨家界、天子山、十里画廊，最后从武陵源门票站出的三日路线具体怎么走最省力，期间景区内环保车和索道交通是否顺畅？"}
{"question": "广州北京路步行街周边评分4.5以上的广汽埃安4S店在哪？"}
{"question": "8月10号从成都到拉萨乘坐火车和飞机哪个更能适应高原反应"}
{"question": "十一假期从广州出发去福建霞浦4天3晚滩涂摄影游，2人，需包车，人均预算2500元以内  "}
{"question": "7月30号长春到乌鲁木齐坐高铁和飞机哪个快"}
{"question": "麻烦设计一条北京一日游公交专线，从颐和园东宫门起，途经圆明园南门、清华大学西门、五道口，终点到奥林匹克公园鸟巢，要求列出每段的公交或地铁线路及预计耗时。"}
{"question": "想从厦门中山路步行街出发，先去南普陀寺拜佛，然后进厦门大学访客中心参观，再去白城沙滩看日落，最后到沙坡尾吃海鲜，有什么不走重复路且避开游客高峰期的建议？"}
{"question": "五一后错峰从深圳飞往甘肃张掖4日丹霞地貌摄影游，2人，需当地拼车，人均预算2800元左右"}
{"question": "请帮我规划一条从成都春熙路出发，先去文殊院上香，再去宽窄巷子喝茶，接着到青羊宫看古银杏，最后去琴台路吃晚饭的步行游览路线，全程尽量控制在5公里以内，怎么走比较合理？"}
{"question": "三亚凤凰机场周边有没有理想汽车的直营店，预算30万左右的车型？"}
